# AI text detection using Qwen3-Embedding-0.6B

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda")

model.max_seq_length = 256

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

In [2]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/Università/deep_learning/progetto_deep_twitter/embeddings_qwen_06'

Mounted at /content/drive


## Cella 3: Caricamento dataset

In [3]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('srikanthgali/ai-text-detection-pile-cleaned')

df_train = pd.DataFrame(dataset['train'])
print(df_train.head())
print(f"Colonne: {df_train.columns.tolist()}")
print(f"Totale campioni: {sum(len(dataset[s]) for s in dataset):,}")

# Controlla la distribuzione delle classi
label_col = 'label' if 'label' in df_train.columns else 'generated'
print(f"\nDistribuzione '{label_col}':")
print(df_train[label_col].value_counts(normalize=True))

README.md:   0%|          | 0.00/9.04k [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/264M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/100M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/99.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/577300 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/72162 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/72164 [00:00<?, ? examples/s]

                                                text  generated source
0  Droplets of water fell from the cracked ceilin...          0  human
1  He couldn't remember how many nights is it bee...          0  human
2  There are lots of great places around you, tha...          1     ai
3  BARACK OBAMA "I am not proposing a big governm...          1     ai
4  Image copyright Getty Images Image caption Env...          0  human
Colonne: ['text', 'generated', 'source']
Totale campioni: 721,626

Distribuzione 'generated':
generated
0    0.500139
1    0.499861
Name: proportion, dtype: float64


## Creazione e save degli embedding (con checkpoint e resume)

In [ ]:
import numpy as np
import os
from tqdm.auto import tqdm
import gc
import torch


gc.collect()
torch.cuda.empty_cache()


# ── Parametri ──────────────────────────────────────────────────────────────
BATCH_SIZE   = 128    # riduci a 32 se vai in OOM
CHUNK_SIZE   = 1000  # campioni per checkpoint su Drive (aumenta se la connessione è veloce)
TEXT_COL     = 'text'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Helpers checkpoint ─────────────────────────────────────────────────────
def chunk_path(split: str, chunk_idx: int, kind: str) -> str:
    """Percorso del file .npy per il chunk `chunk_idx` dello split `split`.
    kind è 'emb' oppure 'lab'.
    """
    return os.path.join(OUTPUT_DIR, f"{split}__{kind}__chunk{chunk_idx:05d}.npy")

def last_completed_chunk(split: str) -> int:
    """Restituisce l'indice dell'ultimo chunk già salvato (-1 se nessuno)."""
    idx = -1
    while os.path.exists(chunk_path(split, idx + 1, 'emb')):
        idx += 1
    return idx

def merge_and_save(split: str, n_chunks: int) -> tuple:
    """Fonde tutti i chunk in due file finali e rimuove i chunk intermedi."""
    emb_parts = [np.load(chunk_path(split, c, 'emb')) for c in range(n_chunks)]
    lab_parts = [np.load(chunk_path(split, c, 'lab')) for c in range(n_chunks)]

    embeddings = np.vstack(emb_parts)
    labels     = np.concatenate(lab_parts)

    np.save(os.path.join(OUTPUT_DIR, f"{split}_embeddings.npy"), embeddings)
    np.save(os.path.join(OUTPUT_DIR, f"{split}_labels.npy"),     labels)

    # Pulizia chunk intermedi
    for c in range(n_chunks):
        os.remove(chunk_path(split, c, 'emb'))
        os.remove(chunk_path(split, c, 'lab'))

    print(f"[{split}] ✅ merge completato → shape {embeddings.shape}")
    return embeddings, labels

# ── Funzione principale con resume ─────────────────────────────────────────
def encode_and_save(split_name: str):
    final_emb = os.path.join(OUTPUT_DIR, f"{split_name}_embeddings.npy")
    final_lab = os.path.join(OUTPUT_DIR, f"{split_name}_labels.npy")
    if os.path.exists(final_emb) and os.path.exists(final_lab):
        print(f"[{split_name}] ⏭  File finali già presenti, skip.")
        return np.load(final_emb), np.load(final_lab)

    # Accediamo direttamente al dataset di Hugging Face (è una mappa, non consuma RAM)
    ds_split = dataset[split_name]
    N = len(ds_split)
    
    label_col = 'label' if 'label' in ds_split.column_names else 'generated'

    resume_chunk = last_completed_chunk(split_name) + 1
    start_sample = resume_chunk * CHUNK_SIZE

    if resume_chunk > 0:
        print(f"[{split_name}] 🔄 Resume dal chunk {resume_chunk} (campione {start_sample:,} / {N:,})")
    else:
        print(f"[{split_name}] 🚀 Avvio da zero ({N:,} campioni)")

    chunk_idx = resume_chunk
    
    # Riduciamo l'uso di memoria: iteriamo direttamente con le funzioni di Hugging Face
    for chunk_start in tqdm(
            range(start_sample, N, CHUNK_SIZE),
            desc=f"{split_name} chunks",
            initial=resume_chunk,
            total=(N + CHUNK_SIZE - 1) // CHUNK_SIZE):

        # Estraiamo SOLO le righe del chunk corrente (operazione istantanea)
        end_idx = min(chunk_start + CHUNK_SIZE, N)
        sub_dataset = ds_split.select(range(chunk_start, end_idx))
        
        # Estraiamo le liste (ora sono piccole, lunghe solo quanto CHUNK_SIZE!)
        chunk_texts = [str(t)[:1500] for t in sub_dataset[TEXT_COL]] # Pre-taglio a 1500 caratteri
        chunk_labels = np.array(sub_dataset[label_col])

        # ENCODING DIRETTO (Sfrutta i 7 secondi visti nel test)
        with torch.inference_mode():
            chunk_matrix = model.encode(
                chunk_texts,
                batch_size=BATCH_SIZE, # Impostalo a 256
                show_progress_bar=False,
                normalize_embeddings=True,
                convert_to_numpy=True
            )
        print(f"salvataggio chunk {chunk_idx}")
        # Salvataggio immediato
        np.save(chunk_path(split_name, chunk_idx, 'emb'), chunk_matrix)
        np.save(chunk_path(split_name, chunk_idx, 'lab'), chunk_labels)

        chunk_idx += 1

    return merge_and_save(split_name, chunk_idx)

# ── Esegui per ogni split disponibile ─────────────────────────────────────
results = {}
for split in dataset.keys():   # 'train', 'test', 'validation' …
    emb, lab = encode_and_save(split)
    results[split] = {"embeddings": emb, "labels": lab}

print("\n✅ Tutti gli embedding sono stati salvati.")

[train] 🔄 Resume dal chunk 16 (campione 16,000 / 577,300)


train chunks:   3%|2         | 16/578 [00:00<?, ?it/s]

salvataggio chunk 16
salvataggio chunk 17
salvataggio chunk 18
salvataggio chunk 19
salvataggio chunk 20
salvataggio chunk 21
salvataggio chunk 22
salvataggio chunk 23
salvataggio chunk 24
salvataggio chunk 25
salvataggio chunk 26
salvataggio chunk 27
salvataggio chunk 28
salvataggio chunk 29
salvataggio chunk 30
salvataggio chunk 31
salvataggio chunk 32
salvataggio chunk 33
salvataggio chunk 34
salvataggio chunk 35
salvataggio chunk 36
salvataggio chunk 37
salvataggio chunk 38
salvataggio chunk 39
salvataggio chunk 40
salvataggio chunk 41
salvataggio chunk 42
salvataggio chunk 43
salvataggio chunk 44
salvataggio chunk 45
salvataggio chunk 46
salvataggio chunk 47
salvataggio chunk 48
salvataggio chunk 49
salvataggio chunk 50
salvataggio chunk 51
salvataggio chunk 52
salvataggio chunk 53
salvataggio chunk 54
salvataggio chunk 55
salvataggio chunk 56
salvataggio chunk 57
salvataggio chunk 58
salvataggio chunk 59
salvataggio chunk 60
salvataggio chunk 61
salvataggio chunk 62
salvataggio c